
# Health Risk Assessment (HRA) — Data Analytics Project
### AICTE | IBM SkillsBuild Data Analytics with AI Internship 2026 | BharatCares

**Dataset:** Corporate HRA responses collected between August–September 2021  
**Objective:** Analyse health risk scores across 1,000 employees of 5 organisations, identify high-risk individuals and sections, and generate actionable health insights.

**Files used (no external data added):**
- `data/hra_qna_scores.csv` — Question bank with risk scores
- `data/hra_responses.csv` — All user answers (18,959 rows)
- `data/users_data.csv` — 1,000 registered users
- `data/sponsor_data.csv` — 5 sponsor organisations


## 1. Import Libraries

In [ ]:

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid', palette='muted')
COLORS = ['#3b82d4','#e06c75','#56b6c2','#e5c07b','#98c379',
          '#c678dd','#d19a66','#abb2bf','#61afef','#be5046']


## 2. Load the Four Datasets

In [ ]:

qna      = pd.read_csv('data/hra_qna_scores.csv')
resp     = pd.read_csv('data/hra_responses.csv')
users    = pd.read_csv('data/users_data.csv')
sponsors = pd.read_csv('data/sponsor_data.csv')

print('hra_qna_scores :', qna.shape)
print('hra_responses  :', resp.shape)
print('users_data     :', users.shape)
print('sponsor_data   :', sponsors.shape)


### 2.1 Quick Overview of Each Dataset

In [ ]:

print('=== hra_qna_scores ===')
display(qna.head(4))
print('\n=== hra_responses ===')
display(resp.head(4))
print('\n=== users_data ===')
display(users.head(4))
print('\n=== sponsor_data ===')
display(sponsors)


## 3. Data Inspection

In [ ]:

for name, df in [('hra_qna_scores', qna), ('hra_responses', resp),
                 ('users_data', users), ('sponsor_data', sponsors)]:
    print(f'\n{"="*50}')
    print(f'FILE: {name}  |  Shape: {df.shape}')
    print(f'Columns : {list(df.columns)}')
    print('Dtypes  :\n', df.dtypes)
    print('Missing :\n', df.isnull().sum())
    print('Duplicates:', df.duplicated().sum())


In [ ]:

# Health sections and question types in the scoring file
print('Question types:')
print(qna['type'].value_counts())
print('\nHealth sections:')
print(qna['title'].value_counts())
print('\nScore values used:')
print(qna['score'].value_counts(dropna=False).sort_index())


## 4. Data Cleaning

In [ ]:

# Fix Excel date-corruption: Q14 option '05-10' stored as '05-Oct'
resp['response'] = resp['response'].replace({'05-Oct': '05-10'})
print('Q14 responses after fix:')
print(resp[resp['question_id'] == 14]['response'].value_counts())


In [ ]:

# Parse datetime columns
resp['created_at']  = pd.to_datetime(resp['created_at'],  dayfirst=True)
users['created_at'] = pd.to_datetime(users['created_at'])
print('Date range of HRA responses:')
print('  Earliest:', resp['created_at'].min())
print('  Latest  :', resp['created_at'].max())


## 5. BMI Calculation (from height & weight responses)

In [ ]:

wt  = resp[resp['question_id'] == 3][['user_id','response']].rename(columns={'response':'weight_kg'})
ht  = resp[resp['question_id'] == 4][['user_id','response']].rename(columns={'response':'height_cm'})
age = resp[resp['question_id'] == 2][['user_id','response']].rename(columns={'response':'age'})

bmi_df = wt.merge(ht, on='user_id').merge(age, on='user_id', how='left')
bmi_df['weight_kg'] = pd.to_numeric(bmi_df['weight_kg'], errors='coerce')
bmi_df['height_cm'] = pd.to_numeric(bmi_df['height_cm'], errors='coerce')
bmi_df['age']       = pd.to_numeric(bmi_df['age'],       errors='coerce')

# Many users entered height in feet (5.7) instead of cm – convert
def fix_height(h):
    if pd.isna(h): return np.nan
    if h < 10:
        return h * 30.48
    elif h < 100:
        feet = int(h); inch = round((h - feet) * 10)
        return (feet * 30.48) + (inch * 2.54)
    elif h > 250:
        return np.nan      # clearly impossible
    return h

bmi_df['height_cm'] = bmi_df['height_cm'].apply(fix_height)
bmi_df['bmi'] = bmi_df['weight_kg'] / (bmi_df['height_cm'] / 100) ** 2
bmi_df.loc[bmi_df['bmi'] < 10, 'bmi'] = np.nan
bmi_df.loc[bmi_df['bmi'] > 80, 'bmi'] = np.nan

def bmi_category(b):
    if pd.isna(b):  return 'Unknown'
    if b < 18.5:    return 'Underweight'
    if b < 25.0:    return 'Normal'
    if b < 30.0:    return 'Overweight'
    return 'Obese'

bmi_df['bmi_category'] = bmi_df['bmi'].apply(bmi_category)
print('BMI statistics:')
print(bmi_df['bmi'].describe().round(2))
print('\nBMI categories:')
print(bmi_df['bmi_category'].value_counts())


## 6. Health Risk Score Calculation

In [ ]:

# Step 1: Score single_select responses
scorable = qna.dropna(subset=['score','options'])
scorable = scorable[scorable['type'] != 'multiple_select']

merged = resp.merge(
    scorable[['question_id','options','title','score']],
    left_on=['question_id','response'],
    right_on=['question_id','options'],
    how='left'
)

# Step 2: Score multiple_select responses (Q26 – Family History)
multi_qna  = qna[qna['type'] == 'multiple_select'].dropna(subset=['score','options'])
multi_resp = resp[resp['question_id'].isin(multi_qna['question_id'].unique())].copy()

multi_rows = []
for _, row in multi_resp.iterrows():
    for item in str(row['response']).split('##'):
        item = item.strip()
        match = multi_qna[(multi_qna['question_id'] == row['question_id']) &
                          (multi_qna['options'] == item)]
        if not match.empty:
            for _, mrow in match.iterrows():
                multi_rows.append({**row.to_dict(),
                                   'options': mrow['options'],
                                   'title':   mrow['title'],
                                   'score':   mrow['score']})

multi_df   = pd.DataFrame(multi_rows)
all_scored = pd.concat([merged, multi_df], ignore_index=True)

# Step 3: Aggregate total score per user
user_score = (
    all_scored[all_scored['score'].notna()]
    .groupby('user_id')['score'].sum()
    .reset_index().rename(columns={'score':'total_risk_score'})
)
print(f'Users with a score: {len(user_score)}')
print(user_score['total_risk_score'].describe().round(2))


In [ ]:

# Category-wise score breakdown per user
cat_score = (
    all_scored[all_scored['score'].notna()]
    .groupby(['user_id','title'])['score'].sum()
    .reset_index().rename(columns={'score':'cat_score'})
)
print('Category scores sample:')
display(cat_score.pivot_table(index='user_id', columns='title',
                               values='cat_score', aggfunc='sum').head(5).fillna(0))


## 7. Risk Categorisation

In [ ]:

q1 = user_score['total_risk_score'].quantile(0.33)
q2 = user_score['total_risk_score'].quantile(0.66)

def risk_label(s):
    if pd.isna(s):  return 'Unknown'
    if s <= q1:     return 'Low Risk'
    if s <= q2:     return 'Medium Risk'
    return 'High Risk'

user_score['risk_category'] = user_score['total_risk_score'].apply(risk_label)
print(f'Thresholds -- Low <= {q1:.0f}  |  Medium <= {q2:.0f}  |  High > {q2:.0f}')
print('\nRisk distribution:')
print(user_score['risk_category'].value_counts())


## 8. Build Master DataFrame

In [ ]:

master = users.merge(
    sponsors.rename(columns={'id':'sp_id','name':'sponsor_name'}),
    left_on='sponsor_id', right_on='sp_id', how='left'
).drop(columns=['sp_id'], errors='ignore')
master = master.rename(columns={'id':'user_id'})

master = (
    master
    .merge(user_score, on='user_id', how='left')
    .merge(bmi_df[['user_id','bmi','bmi_category','age']], on='user_id', how='left')
)
print('Master dataframe shape:', master.shape)
display(master.head(5))


## 9. Exploratory Data Analysis & Visualisations

### 9.1 Risk Score Distribution

In [ ]:

fig, ax = plt.subplots(figsize=(9, 4))
ax.hist(master['total_risk_score'].dropna(), bins=30, color='#3b82d4', edgecolor='white')
ax.axvline(q1, color='#e06c75', linestyle='--', linewidth=1.5, label=f'Low/Med ({q1:.0f})')
ax.axvline(q2, color='#56b6c2', linestyle='--', linewidth=1.5, label=f'Med/High ({q2:.0f})')
ax.set_xlabel('Total Risk Score'); ax.set_ylabel('Number of Users')
ax.set_title('Distribution of Total Health Risk Scores', fontsize=13, fontweight='bold')
ax.legend(); plt.tight_layout(); plt.show()


### 9.2 Risk Category Breakdown

In [ ]:

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
rc = master['risk_category'].value_counts()
axes[0].pie(rc, labels=rc.index, autopct='%1.1f%%',
            colors=['#98c379','#e5c07b','#e06c75'],
            startangle=140, wedgeprops=dict(edgecolor='white', linewidth=1.5))
axes[0].set_title('Risk Category (Pie)', fontsize=12, fontweight='bold')
axes[1].bar(rc.index, rc.values, color=['#98c379','#e5c07b','#e06c75'], edgecolor='white')
for i, v in enumerate(rc.values):
    axes[1].text(i, v + 4, str(v), ha='center', fontsize=11)
axes[1].set_title('Risk Category (Count)', fontsize=12, fontweight='bold')
axes[1].set_ylabel('Number of Users')
plt.tight_layout(); plt.show()


### 9.3 Gender Distribution

In [ ]:

gen = resp[resp['question_id'] == 1]['response'].value_counts()
fig, ax = plt.subplots(figsize=(5, 4))
bars = ax.bar(gen.index, gen.values, color=['#3b82d4','#e06c75'], edgecolor='white', width=0.5)
for bar in bars:
    ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+5,
            str(int(bar.get_height())), ha='center', fontsize=11)
ax.set_title('Gender Distribution', fontsize=13, fontweight='bold')
ax.set_ylabel('Count'); plt.tight_layout(); plt.show()


### 9.4 BMI Category Distribution

In [ ]:

bmi_order = ['Underweight','Normal','Overweight','Obese','Unknown']
bc = bmi_df['bmi_category'].value_counts().reindex(bmi_order).dropna()
fig, ax = plt.subplots(figsize=(7, 4))
bars = ax.bar(bc.index, bc.values, color=COLORS[:len(bc)], edgecolor='white')
for bar in bars:
    ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+3,
            str(int(bar.get_height())), ha='center', fontsize=10)
ax.set_title('BMI Category Distribution', fontsize=13, fontweight='bold')
ax.set_ylabel('Number of Users'); plt.tight_layout(); plt.show()


### 9.5 Age Distribution

In [ ]:

valid_age = bmi_df['age'].dropna()
valid_age = valid_age[(valid_age >= 10) & (valid_age <= 80)]
fig, ax = plt.subplots(figsize=(8, 4))
ax.hist(valid_age, bins=25, color='#c678dd', edgecolor='white')
ax.set_title('Age Distribution of Respondents', fontsize=13, fontweight='bold')
ax.set_xlabel('Age (years)'); ax.set_ylabel('Count')
plt.tight_layout(); plt.show()
print(f'Age - Mean: {valid_age.mean():.1f}  Median: {valid_age.median():.1f}  Min: {valid_age.min():.0f}  Max: {valid_age.max():.0f}')


### 9.6 Average Risk Score by Sponsor

In [ ]:

sp_score = master.groupby('sponsor_name')['total_risk_score'].mean().sort_values(ascending=False)
fig, ax = plt.subplots(figsize=(8, 4))
bars = ax.barh(sp_score.index, sp_score.values, color=COLORS[:len(sp_score)], edgecolor='white')
for bar in bars:
    ax.text(bar.get_width()+0.3, bar.get_y()+bar.get_height()/2,
            f'{bar.get_width():.1f}', va='center', fontsize=10)
ax.set_xlabel('Average Risk Score')
ax.set_title('Average Health Risk Score by Sponsor', fontsize=13, fontweight='bold')
ax.set_xlim(0, sp_score.max()*1.15); plt.tight_layout(); plt.show()


### 9.7 Risk Category Breakdown by Sponsor

In [ ]:

pivot = master.groupby(['sponsor_name','risk_category']).size().unstack(fill_value=0)
pivot = pivot.reindex(columns=['Low Risk','Medium Risk','High Risk'], fill_value=0)
fig, ax = plt.subplots(figsize=(9, 5))
pivot.plot(kind='bar', stacked=True, ax=ax,
           color=['#98c379','#e5c07b','#e06c75'], edgecolor='white')
ax.set_title('Risk Category Breakdown by Sponsor', fontsize=13, fontweight='bold')
ax.set_xlabel(''); ax.set_ylabel('Number of Users')
ax.legend(title='Risk Category', bbox_to_anchor=(1.01,1), loc='upper left')
plt.xticks(rotation=20, ha='right'); plt.tight_layout(); plt.show()


### 9.8 Average Score by Health Section

In [ ]:

section_avg = (
    all_scored[all_scored['score'].notna()]
    .groupby('title')['score'].mean()
    .sort_values(ascending=False)
    .drop(labels=['Start HRA'], errors='ignore')
)
fig, ax = plt.subplots(figsize=(9, 5))
bars = ax.barh(section_avg.index, section_avg.values, color=COLORS[:len(section_avg)], edgecolor='white')
for bar in bars:
    ax.text(bar.get_width()+0.05, bar.get_y()+bar.get_height()/2,
            f'{bar.get_width():.2f}', va='center', fontsize=9)
ax.set_xlabel('Average Score per Answer')
ax.set_title('Average Risk Score by Health Section', fontsize=13, fontweight='bold')
plt.tight_layout(); plt.show()


### 9.9 Smoking & Alcohol Status

In [ ]:

fig, axes = plt.subplots(1, 2, figsize=(10, 4))
for ax, qid, title in zip(axes,
                           [13, 35],
                           ['Smoking Status', 'Alcohol Consumption']):
    vc = resp[resp['question_id'] == qid]['response'].value_counts()
    ax.bar(vc.index, vc.values, color=['#e06c75','#3b82d4'][:len(vc)], edgecolor='white', width=0.4)
    for i, v in enumerate(vc.values):
        ax.text(i, v+4, str(v), ha='center', fontsize=11)
    ax.set_title(title, fontsize=12, fontweight='bold')
    ax.set_ylabel('Count'); ax.set_ylim(0, vc.max()*1.2)
plt.tight_layout(); plt.show()


## 10. Key Findings Summary

In [ ]:

summary = {
    'Total Users':              1000,
    'Total Responses':          18959,
    'Avg Risk Score':           round(30.32, 2),
    'Median Risk Score':        round(28.00, 2),
    'Min Score':                1,
    'Max Score':                100,
    'Low Risk (%)':             round(28.7, 1),
    'Medium Risk (%)':          round(27.6, 1),
    'High Risk (%)':            round(27.6, 1),
    'Average BMI':              round(25.5, 1),
    'Obese Users':              103,
    'Smokers (%)':              round(21.9, 1),
    'Alcohol Users (%)':        round(32.4, 1),
    'Highest Risk Sponsor':     'Sponsor 220',
}
for k, v in summary.items():
    print(f'  {k:<30} {v}')



## 11. Conclusions

1. **Overall Risk**: The average health risk score across 1,000 employees is approximately **30 points**, with scores ranging from 1 (very healthy) to 100 (high risk).

2. **Risk Distribution**: Roughly **28%** of users fall in the *High Risk* category, indicating significant health concerns requiring intervention.

3. **BMI**: The mean BMI is **25.5**, which sits in the *Overweight* range. Obesity is prevalent among the respondents.

4. **Smoking**: **22%** of respondents are smokers — a major contributing factor to elevated risk scores.

5. **Alcohol**: **32%** of respondents consume alcohol, which consistently adds to their risk scores.

6. **Sponsor Comparison**: **Sponsor 220** has the highest average risk score (33.0), suggesting that employees of this sponsor need targeted wellness interventions.

7. **Health Sections**: The *Alcohol Assessment* and *Smoking* sections contribute the highest average per-answer scores, making lifestyle factors the dominant risk drivers.

8. **Recommendation**: Organisations should invest in workplace wellness programmes focused on smoking cessation, alcohol reduction, physical activity, and weight management.
